# 01 Data Acquisition

Notebook-first walkthrough for branch `02-data-ingestion`. The goal is to inspect the structured ingestion flow before relying on the script entry point.

## 1. Configure the Run

Use fixture mode for deterministic local development. Switch to live mode when you want fresh yfinance and SEC EDGAR data.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from constants import INGESTION_MODE_FIXTURE  # noqa: E402
from ingestion.pipeline import DEFAULT_FIXTURE_PATH, run_ingestion  # noqa: E402

MODE = INGESTION_MODE_FIXTURE
DB_PATH = PROJECT_ROOT / "data" / "processed" / "notebook_ingestion.sqlite"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
TICKERS = ["MSFT", "NVDA"]
START_DATE = "2024-01-02"
END_DATE = "2024-01-05"
REFRESH_UNIVERSE = False

## 2. Inspect the Fixture Payload

Fixture mode uses committed test data, but it still goes through the same normalization and SQLite persistence code as live mode.

In [2]:
import json

fixture_payload = json.loads(DEFAULT_FIXTURE_PATH.read_text(encoding="utf-8"))
fixture_payload.keys()

dict_keys(['companies', 'index_constituents', 'prices', 'sec_company_facts'])

In [3]:
fixture_payload["companies"]

[{'company_id': 'MSFT',
  'ticker': 'MSFT',
  'name': 'Microsoft Corp',
  'sector': 'Information Technology',
  'industry': 'Systems Software',
  'cik': '0000789019'},
 {'company_id': 'NVDA',
  'ticker': 'NVDA',
  'name': 'NVIDIA Corp',
  'sector': 'Information Technology',
  'industry': 'Semiconductors',
  'cik': '0001045810'}]

## 3. Run the Shared Ingestion Pipeline

The notebook intentionally calls `run_ingestion()` rather than duplicating the implementation. This keeps the notebook, tests, and future script behavior aligned.

In [4]:
result = run_ingestion(
    mode=MODE,
    db_path=DB_PATH,
    raw_data_dir=RAW_DATA_DIR,
    start_date=START_DATE,
    end_date=END_DATE,
    tickers=TICKERS,
    refresh_universe=REFRESH_UNIVERSE,
)
result

IngestionResult(run_id='ee06151c-a115-42e6-971d-a8b5bdc27886', db_path=PosixPath('/Users/nickcruickshank/Projects/ai-investment-decision-support/data/processed/notebook_ingestion.sqlite'), source_summary={'companies': 2, 'index_constituents': 2, 'price_bars': 4, 'fundamental_facts': 4, 'raw_artifacts': 4})

## 4. Check SQLite Outputs

Run lightweight sanity queries against the local database before promoting the same workflow to the CLI script.

In [5]:
import sqlite3

connection = sqlite3.connect(DB_PATH)
connection.row_factory = sqlite3.Row

tables = connection.execute(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"
).fetchall()
[row["name"] for row in tables]

['companies',
 'fundamental_facts',
 'index_constituents',
 'ingestion_runs',
 'price_bars',
 'raw_artifacts']

In [6]:
for table_name in [
    "companies",
    "index_constituents",
    "price_bars",
    "fundamental_facts",
    "raw_artifacts",
]:
    count = connection.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"{table_name}: {count}")

companies: 2
index_constituents: 2
price_bars: 4
fundamental_facts: 4
raw_artifacts: 4


In [7]:
connection.execute(
    """
    SELECT c.ticker, c.name, c.cik, ic.weight, ic.as_of_date
    FROM companies AS c
    JOIN index_constituents AS ic ON c.company_id = ic.company_id
    ORDER BY ic.rank
    """
).fetchall()

[<sqlite3.Row at 0x1126e98d0>, <sqlite3.Row at 0x1126e93f0>]

## 5. Script Equivalent

Once the notebook flow looks right, the same path can be run from `scripts/ingest_data.py`.

In [8]:
!python ../scripts/ingest_data.py --mode fixture --tickers MSFT NVDA

Ingestion run completed: 3d90be65-b605-4e57-b519-7ffdb8415062
SQLite database: /Users/nickcruickshank/Projects/ai-investment-decision-support/data/processed/investment_data.sqlite
companies: 2
fundamental_facts: 4
index_constituents: 2
price_bars: 4
raw_artifacts: 4


## 6. Personal Exploration

In [21]:
import pandas as pd

In [40]:
import sqlite3

connection = sqlite3.connect(DB_PATH)
connection.row_factory = sqlite3.Row
DB_PATH = PROJECT_ROOT / "data" / "processed" / "investment_data.sqlite"

tables = connection.execute(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"
).fetchall()
[row["name"] for row in tables]

['companies',
 'fundamental_facts',
 'index_constituents',
 'ingestion_runs',
 'price_bars',
 'raw_artifacts']

In [47]:
# show companies
query = """
select 
    *
from 
    companies
"""
df = pd.read_sql_query(query, connection)
print(df.shape)
df.head()

(10, 9)


,company_id,ticker,name,sector,industry,cik,is_active,created_at,updated_at
0,MSFT,MSFT,Microsoft Corporation,Technology,Software - Infrastructure,0000789019,1,2026-07-31T18:06:05.346907+00:00,2026-07-31T18:22:12.294086+00:00
1,NVDA,NVDA,NVIDIA Corporation,Technology,Semiconductors,0001045810,1,2026-07-31T18:06:05.346907+00:00,2026-07-31T18:22:12.294086+00:00
2,AAPL,AAPL,Apple Inc.,Technology,Consumer Electronics,0000320193,1,2026-07-31T18:22:12.294086+00:00,2026-07-31T18:22:12.294086+00:00
3,AMZN,AMZN,"Amazon.com, Inc.",Consumer Cyclical,Internet Retail,0001018724,1,2026-07-31T18:22:12.294086+00:00,2026-07-31T18:22:12.294086+00:00
4,GOOGL,GOOGL,Alphabet Inc.,Communication Services,Internet Content & Information,0001652044,1,2026-07-31T18:22:12.294086+00:00,2026-07-31T18:22:12.294086+00:00


In [46]:
# show fundamental_facts
query = """
select 
    *
from 
    fundamental_facts
"""
df = pd.read_sql_query(query, connection)
print(df.shape)
df.head()

(7306, 14)


,ticker,cik,taxonomy,concept,label,unit,value,period_start,period_end,fiscal_year,fiscal_period,form,filed_at,source
0,MSFT,0000789019,us-gaap,Revenues,Revenues,USD,2.451220e+11,2023-07-01,2024-06-30,2024.0,FY,10-K,2024-07-30,sec_edgar
1,MSFT,0000789019,us-gaap,NetIncomeLoss,Net Income (Loss) Attributable to Parent,USD,8.813600e+10,2023-07-01,2024-06-30,2024.0,FY,10-K,2024-07-30,sec_edgar
2,NVDA,0001045810,us-gaap,Revenues,Revenues,USD,1.304970e+11,2024-01-29,2025-01-26,2025.0,FY,10-K,2025-02-26,sec_edgar
3,NVDA,0001045810,us-gaap,Assets,Assets,USD,1.116010e+11,NaN,2025-01-26,2025.0,FY,10-K,2025-02-26,sec_edgar
4,NVDA,0001045810,us-gaap,Revenues,Revenues,USD,4.097860e+09,2007-01-29,2008-01-27,2010.0,FY,10-K,2010-03-18,sec_edgar


In [48]:
# show index_constituents
query = """
select 
    *
from 
    index_constituents
"""
df = pd.read_sql_query(query, connection)
print(df.shape)
df.head()

(10, 7)


,index_symbol,company_id,ticker,weight,rank,as_of_date,source
0,SP500,NVDA,NVDA,7.44,1,2026-07-30,pinned_universe
1,SP500,MSFT,MSFT,4.21,3,2026-07-30,pinned_universe
2,SP500,AAPL,AAPL,7.27,2,2026-07-30,pinned_universe
3,SP500,AMZN,AMZN,3.71,4,2026-07-30,pinned_universe
4,SP500,GOOGL,GOOGL,2.99,5,2026-07-30,pinned_universe


In [49]:
query = """
select 
    *
from 
    price_bars
"""
df = pd.read_sql_query(query, connection)
print(df.shape)
df.head()

(6450, 9)


,ticker,date,open,high,low,close,adj_close,volume,source
0,MSFT,2024-01-02,373.859985,375.899994,366.769989,370.869995,363.801453,25258600.0,yfinance
1,MSFT,2024-01-03,369.010010,373.260010,368.510010,370.600006,363.536621,23083500.0,yfinance
2,NVDA,2024-01-02,49.243999,49.294998,47.595001,48.167999,48.082535,411254000.0,yfinance
3,NVDA,2024-01-03,47.485001,48.183998,47.320000,47.569000,47.484592,320896000.0,yfinance
4,META,2024-01-02,351.320007,353.160004,340.010010,346.290009,343.275513,19042200.0,yfinance


In [50]:
# show raw_artifacts
query = """
select 
    *
from 
    raw_artifacts
"""
df = pd.read_sql_query(query, connection)
print(df.shape)
df.head()

(30, 9)


,artifact_id,run_id,source,ticker,artifact_type,source_url,local_path,content_hash,fetched_at
0,37fb18f0-bd08-43c7-a847-4733a5bab2e9,4656f3ca-f3ab-46e4-a774-7cc33dac8ffe,pinned_universe,NaN,fixture_universe,None,/Users/nickcruickshank/Projects/ai-investment-...,86ffc25bc9124a46e3f21916a534aa76115a6095d01a2a...,2026-07-31T18:06:05.345899+00:00
1,eb0322b6-d5dd-4a7e-94a2-7c683be531ff,4656f3ca-f3ab-46e4-a774-7cc33dac8ffe,yfinance,NaN,fixture_prices,None,/Users/nickcruickshank/Projects/ai-investment-...,102e0c0e4ca7af8a2659befba3d05791661b86db50c853...,2026-07-31T18:06:05.346386+00:00
2,530a95db-6360-4584-aa57-cb4b9378c8be,4656f3ca-f3ab-46e4-a774-7cc33dac8ffe,sec_edgar,MSFT,fixture_company_facts,None,/Users/nickcruickshank/Projects/ai-investment-...,b53a2b47f376d4f5ef05bd404c47168580529861a71217...,2026-07-31T18:06:05.346685+00:00
3,6173548e-31c4-401d-967c-4770f3f4b133,4656f3ca-f3ab-46e4-a774-7cc33dac8ffe,sec_edgar,NVDA,fixture_company_facts,None,/Users/nickcruickshank/Projects/ai-investment-...,da5cb9f3c0df8d5f2b9dc397106addfef201094a852450...,2026-07-31T18:06:05.346797+00:00
4,c2c16d9c-88fe-4d06-8b50-714f69f99419,a5d26006-5ea3-45c9-8782-2e50f7a87d45,pinned_universe,NaN,fixture_universe,None,/Users/nickcruickshank/Projects/ai-investment-...,86ffc25bc9124a46e3f21916a534aa76115a6095d01a2a...,2026-07-31T18:14:21.643120+00:00


I need more than just two random days